# Datathon 2026 Final Notebook

## Track
Sustainability & Critical Infrastructure

## Project Title
Global CO2 Emissions Change Prediction from Industry/Sector GDP Changes

## Team
- Add team name
- Add team member names
- Add contact emails

## 1) Problem Statement and Hypothesis

### Problem statement
Can year-over-year GDP changes across sectors/industries explain and predict year-over-year CO2 emissions changes at a global trend level?

### Hypothesis
If high-emission sectors (especially industrial and transport-related sectors) increase in GDP share/value, CO2 emissions tend to increase; if they contract, CO2 emissions tend to decrease.

## 2) Dataset Description

This project uses two datasets in the `data/` folder:

1. `ghg-emissions-by-sector.csv`
   - Country-year greenhouse gas emissions by sector
2. `Download-GDPcurrent-USD-countries.xlsx`
   - Country-year GDP indicators, including sector indicators marked with `(ISIC)`

Target used for modeling:
- Year-over-year change in selected CO2-related emissions sum

Predictors used for modeling:
- Year-over-year GDP changes of selected `(ISIC)` sector indicators

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


DATA_DIR = Path('data')
EMISSIONS_CSV = DATA_DIR / 'ghg-emissions-by-sector.csv'
GDP_XLSX = DATA_DIR / 'Download-GDPcurrent-USD-countries.xlsx'


def load_and_transform_emissions(csv_path: Path) -> pd.DataFrame:
    emissions = pd.read_csv(csv_path)
    emissions['building_construction_manufacturing'] = (
        emissions['Buildings'] + emissions['Manufacturing and construction']
    )

    drop_cols = [
        'Fugitive emissions',
        'Other fuel combustion',
        'Buildings',
        'Manufacturing and construction',
    ]
    emissions = emissions.drop(columns=drop_cols)
    emissions = emissions.rename(columns={'Entity': 'Country'})
    return emissions


def load_isic_gdp_features(gdp_xlsx: Path) -> pd.DataFrame:
    gdp_raw = pd.read_excel(
        gdp_xlsx, sheet_name='Download-GDPcurrent-USD-countri', header=2
    )
    year_cols = [c for c in gdp_raw.columns if isinstance(c, (int, float))]
    long_df = gdp_raw[['Country', 'IndicatorName'] + year_cols].melt(
        id_vars=['Country', 'IndicatorName'], var_name='Year', value_name='value'
    )
    long_df['Year'] = long_df['Year'].astype(int)

    isic = long_df[long_df['IndicatorName'].str.contains(r'\(ISIC', na=False)].copy()
    isic = isic.pivot_table(
        index=['Country', 'Year'],
        columns='IndicatorName',
        values='value',
        aggfunc='first',
    ).reset_index()

    # Remove broad bucket and avoid manufacturing double-counting.
    drop_col = 'Other Activities (ISIC J-P)'
    ce_col = 'Mining, Manufacturing, Utilities (ISIC C-E)'
    d_col = 'Manufacturing (ISIC D)'
    derived_col = 'Mining and Utilities (derived from ISIC C-E minus ISIC D)'

    if drop_col in isic.columns:
        isic = isic.drop(columns=[drop_col])

    if ce_col in isic.columns and d_col in isic.columns:
        isic[derived_col] = isic[ce_col] - isic[d_col]
        isic = isic.drop(columns=[ce_col])

    return isic


def build_change_dataset(emissions_csv: Path, gdp_xlsx: Path) -> tuple[pd.DataFrame, pd.Series]:
    emissions = load_and_transform_emissions(emissions_csv)
    isic = load_isic_gdp_features(gdp_xlsx)
    merged = emissions.merge(isic, on=['Country', 'Year'], how='inner')

    emission_feature_cols = [
        c for c in emissions.columns if c not in ['Country', 'Code', 'Year']
    ]
    merged['co2_total_selected'] = merged[emission_feature_cols].sum(axis=1)

    isic_cols = [c for c in isic.columns if c not in ['Country', 'Year']]
    merged = merged.sort_values(['Country', 'Year'])

    change_cols = []
    for col in isic_cols:
        new_col = f'gdp_change_{col}'
        merged[new_col] = merged.groupby('Country')[col].pct_change(fill_method=None)
        change_cols.append(new_col)

    merged['co2_change_target'] = merged.groupby('Country')['co2_total_selected'].pct_change(
        fill_method=None
    )

    model_df = merged[change_cols + ['co2_change_target']].replace([np.inf, -np.inf], np.nan)
    model_df = model_df.dropna(subset=['co2_change_target'])
    model_df = model_df.fillna(0.0)

    X = model_df[change_cols]
    y = model_df['co2_change_target']
    return X, y


def train_model(X: pd.DataFrame, y: pd.Series) -> tuple[Pipeline, dict]:
    numeric_cols = list(X.columns)
    preprocessor = ColumnTransformer(
        transformers=[('num', SimpleImputer(strategy='median'), numeric_cols)]
    )
    model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
    pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = {
        'rows_used': int(len(X)),
        'features_used': int(len(X.columns)),
        'mae_change': float(mean_absolute_error(y_test, y_pred)),
        'r2': float(r2_score(y_test, y_pred)),
    }
    return pipeline, metrics

## 3) Data Cleaning and Preprocessing

Business rules applied:
- Combine `Buildings` + `Manufacturing and construction`
- Exclude `Fugitive emissions` and `Other fuel combustion`
- Keep only GDP indicators with `(ISIC)` in indicator names
- Remove `Other Activities (ISIC J-P)`
- Replace `Mining, Manufacturing, Utilities (ISIC C-E)` with a derived feature:
  - `Mining and Utilities = (ISIC C-E) - Manufacturing (ISIC D)`

Modeling transformation:
- Convert both GDP sector values and emissions totals to year-over-year percentage change by country
- Pool all country-year rows for training a country-agnostic trend model

In [ ]:
emissions = load_and_transform_emissions(EMISSIONS_CSV)
isic = load_isic_gdp_features(GDP_XLSX)
X, y = build_change_dataset(EMISSIONS_CSV, GDP_XLSX)

print('Emissions shape:', emissions.shape)
print('ISIC feature shape:', isic.shape)
print('Model X shape:', X.shape)
print('Model y shape:', y.shape)
print('\nModel features:')
for c in X.columns:
    print('-', c)

## 4) Exploratory Data Analysis (Charts + Interpretation)

The following charts inspect the distribution of feature changes and target CO2 changes.

Interpretation guide:
- Wide tails indicate volatile year-over-year changes
- Heavy concentration around zero means many years have modest changes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target distribution
y.clip(-2, 2).hist(bins=60, ax=axes[0])
axes[0].set_title('Distribution: CO2 change target (clipped)')
axes[0].set_xlabel('Year-over-year CO2 change ratio')

# Example feature distribution
example_col = X.columns[0]
X[example_col].clip(-2, 2).hist(bins=60, ax=axes[1])
axes[1].set_title(f'Distribution: {example_col} (clipped)')
axes[1].set_xlabel('Year-over-year GDP change ratio')

plt.tight_layout()
plt.show()

## 5) Statistical Testing and/or Model Approach

Model used:
- `RandomForestRegressor` in a preprocessing pipeline
- Median imputation for missing values

Rationale:
- Handles nonlinear relationships between sector GDP changes and CO2 change target
- Robust to mixed scale and non-normal feature behavior

In [ ]:
model, metrics = train_model(X, y)
metrics

## 6) Results and Conclusions

- The model produces a global trend estimate of CO2 change from GDP sector changes.
- It is designed for scenario testing (direction and rough magnitude), not country-specific policy decisions.
- Current baseline performance indicates limited predictive power, so outputs should be interpreted with caution.

## 7) Limitations and Future Work

### Limitations
- Mining and utilities are derived indirectly from `(ISIC C-E) - (ISIC D)` due to source granularity.
- Year-over-year change targets are noisy and sensitive to shocks.
- Model does not include policy, energy mix, or technology adoption variables.

### Future work
- Add lagged and rolling trend features.
- Compare with regularized linear models and gradient boosting.
- Include additional external variables (energy price indices, renewables share, policy indicators).

## Reproducibility

This notebook is self-contained for data loading, preprocessing, and model training.

1. Install dependencies:
   - `pip install -r requirements.txt`
2. Open and run this notebook from top to bottom.
3. (Optional) Use project scripts for app/demo:
   - `python predict_co2_change.py --baseline-co2 1000000`
   - `streamlit run app.py`

## Dataset Citations (MLA 8)

Our World in Data. *GHG Emissions by Sector*. Our World in Data, https://ourworldindata.org/grapher/ghg-emissions-by-sector. Accessed 29 Mar. 2026.

United Nations Statistics Division. *GDP/breakdown at current prices in US Dollars (all countries)*. UNdata, https://data.un.org/. Accessed 29 Mar. 2026.

AI-Community-SBU. *Datathon-2026 Submission Guidelines*. GitHub, https://github.com/AI-Community-SBU/Datathon-2026/tree/main/submission-guidelines. Accessed 29 Mar. 2026.